In [1]:
import pandas as pd
import numpy as np

DATA_FILE      = "DISSERTATION_FULL_DATA_FINAL.xlsx"
FILL_THRESHOLD = 5             # forward fill gaps up to one trading week
SAMPLE_START   = "2010-01-01"
SAMPLE_END     = "2024-12-31"
MIN_FIRMS      = 5             # minimum constituents to compute a composite

all_sheets = pd.read_excel(DATA_FILE, sheet_name=None, header=None)

In [2]:
# Six sectors: XLE has 6 firms, EXE excluded.
SECTOR_CONFIG = {
    "XLF 5Y FINANCIALS": {"code": "XLF", "tickers": {
        "JPMORGAN": "JPM", "JP MORGAN": "JPM", "BERKSHIRE": "BRK",
        "BANK OF AMERICA": "BAC", "BOFA": "BAC", "GOLDMAN": "GS",
        "MORGAN STANLEY": "MS", "WELLS FARGO": "WFC", "CITIGROUP": "C",
        "CITI": "C", "AMERICAN EXPRESS": "AXP", "CAPITAL ONE": "COF", "CHUBB": "CB"}},
    "XLI 5Y INDUSTRIALS": {"code": "XLI", "tickers": {
        "RAYTHEON": "RTX", "EATON": "ETN", "DEERE": "DE", "LOCKHEED": "LMT",
        "GENERAL DYNAMICS": "GD", "3M": "MMM", "HONEYWELL": "HON",
        "BOEING": "BA", "CATERPILLAR": "CAT", "UNION PACIFIC": "UNP"}},
    "XLV 5Y HEALTHCARE": {"code": "XLV", "tickers": {
        "ELI LILLY": "LLY", "LILLY": "LLY", "JOHNSON": "JNJ", "J&J": "JNJ",
        "UNITEDHEALTH": "UNH", "UNITED HEALTH": "UNH", "MERCK": "MRK",
        "CVS": "CVS", "MEDTRONIC": "MDT", "HUMANA": "HUM", "AMGEN": "AMGN",
        "PFIZER": "PFE", "BRISTOL": "BMY"}},
    "XLY 5Y CONSUMER DISCRETIONARY": {"code": "XLY", "tickers": {
        "TJX": "TJX", "MARRIOTT": "MAR", "NIKE": "NKE", "EXPEDIA": "EXPE",
        "ROYAL CARIBBEAN": "RCL", "ROYAL": "RCL", "MGM": "MGM",
        "CARNIVAL": "CCL", "HOME DEPOT": "HD", "MCDONALD": "MCD",
        "LOWE'S": "LOW", "LOWES": "LOW"}},
    "XLK 5Y TECHNOLOGY": {"code": "XLK", "tickers": {
        "MICROSOFT": "MSFT", "INTEL": "INTC", "TEXAS INSTRUMENTS": "TXN",
        "ORACLE": "ORCL", "SEAGATE": "STX", "CORNING": "GLW", "NXP": "NXPI",
        "MOTOROLA": "MSI", "IBM": "IBM", "APPLE": "AAPL"}},
    "XLE 5Y ENERGY": {"code": "XLE", "tickers": {
        "EXXON": "XOM", "CHEVRON": "CVX", "WILLIAMS": "WMB", "EOG": "EOG",
        "ONEOK": "OKE", "OCCIDENTAL": "OXY"}},}


def find_data_start(df):
    for i in range(8):
        try:
            if pd.notna(pd.to_datetime(df.iloc[i, 0])):
                return i
        except (ValueError, TypeError):
            pass
    raise ValueError("Could not locate data start.")


def map_columns(df, ticker_map):
    header = [str(df.iloc[0, c]).strip().upper() for c in range(df.shape[1])]
    keys_by_length = sorted(ticker_map, key=len, reverse=True)
    col_to_ticker, used = {}, set()
    for col, name in enumerate(header):
        if name in ("", "NAN", "DATE", "NONE"):
            continue
        for key in keys_by_length:
            ticker = ticker_map[key]
            if ticker not in used and key.upper() in name:
                col_to_ticker[col] = ticker
                used.add(ticker)
                break
    return col_to_ticker

In [3]:
# Forward fill gaps up to one week, leave longer gaps missing, then average across firms to an equal weighted composite. 
# Days with fewer than MIN_FIRMS contributors are set missing.
sector_composites = {}
for sheet, cfg in SECTOR_CONFIG.items():
    if sheet not in all_sheets:
        continue
    sector = cfg["code"]
    raw = all_sheets[sheet].dropna(axis=1, how="all")
    start = find_data_start(raw)
    col_map = map_columns(raw, cfg["tickers"])
    rows = raw.iloc[start:]
    dates = pd.to_datetime(rows.iloc[:, 0], errors="coerce")
    rows, dates = rows[dates.notna()], dates[dates.notna()]
    firm = {}
    for col, ticker in col_map.items():
        s = pd.to_numeric(rows.iloc[:, col], errors="coerce")
        s.index = dates
        firm[ticker] = s
    raw_df = pd.DataFrame(firm).sort_index().loc[SAMPLE_START:SAMPLE_END]
    clean = raw_df.apply(lambda s: s.ffill(limit=FILL_THRESHOLD))
    n_valid = clean.notna().sum(axis=1)
    composite = clean.mean(axis=1, skipna=True)
    composite[n_valid < MIN_FIRMS] = np.nan
    sector_composites[sector] = pd.DataFrame({
        "Sector":         sector,
        "CDS_level":      composite,
        "CDS_change_5d":  composite.diff(5),
        "N_constituents": n_valid,})

In [4]:
# Sector RV from ETF prices (Corsi, 2009): RV_daily = 252 * r^2 (annualised
# squared log return); weekly/monthly are backward rolling means.
ETF_COLS = {"Financials": "XLF", "Energy": "XLE", "Technology": "XLK",
            "Healthcare": "XLV", "Industrials": "XLI", "Consumer Discretionary": "XLY"}

etf = all_sheets["ETF PRICE DATA"].dropna(axis=1, how="all")
estart = find_data_start(etf)
names = [str(etf.iloc[0, c]).strip() for c in range(etf.shape[1])]
erows = etf.iloc[estart:]
edates = pd.to_datetime(erows.iloc[:, 0], errors="coerce")
erows, edates = erows[edates.notna()], edates[edates.notna()]
prices = {}
for c, nm in enumerate(names):
    if nm in ETF_COLS:
        p = pd.to_numeric(erows.iloc[:, c], errors="coerce")
        p.index = edates
        prices[ETF_COLS[nm]] = p
price_df = pd.DataFrame(prices).sort_index()
log_ret = np.log(price_df).diff()

all_rv = {}
for sector in ETF_COLS.values():
    if sector not in log_ret.columns:
        continue
    rv_d = (log_ret[sector] ** 2) * 252
    all_rv[sector] = pd.DataFrame({
        "RV_daily":   rv_d,
        "RV_weekly":  rv_d.rolling(5,  min_periods=5).mean(),
        "RV_monthly": rv_d.rolling(21, min_periods=21).mean(),})

# Macro controls: VIX, Treasury yields and Baa spread.
mac = all_sheets["MACRO CONTROLS"].dropna(axis=1, how="all")
mstart = find_data_start(mac)
mrows = mac.iloc[mstart:]
mdates = pd.to_datetime(mrows.iloc[:, 0], errors="coerce")
mrows, mdates = mrows[mdates.notna()], mdates[mdates.notna()]
macro = pd.DataFrame({
    "VIX":   pd.to_numeric(mrows.iloc[:, 1], errors="coerce").values,
    "US2Y":  pd.to_numeric(mrows.iloc[:, 2], errors="coerce").values,
    "US10Y": pd.to_numeric(mrows.iloc[:, 3], errors="coerce").values,
}, index=pd.to_datetime(mdates.values)).sort_index().loc[SAMPLE_START:SAMPLE_END]
macro["Yield_slope"] = macro["US10Y"] - macro["US2Y"]

baa = pd.read_excel("BAA10Y.xlsx", sheet_name="Daily", header=0)
baa.columns = ["Date", "BAA10Y"]
baa["Date"] = pd.to_datetime(baa["Date"])
baa["BAA10Y"] = pd.to_numeric(baa["BAA10Y"], errors="coerce")
baa = baa.set_index("Date").sort_index().loc[SAMPLE_START:SAMPLE_END]

macro = macro.join(baa, how="left")
for col in ["VIX", "Yield_slope", "BAA10Y"]:
    macro[col] = macro[col].ffill(limit=FILL_THRESHOLD)

In [5]:
frames = []
for sector in ["XLF", "XLI", "XLV", "XLY", "XLK", "XLE"]:
    cds = sector_composites[sector].drop(columns=["Sector"])
    rv  = all_rv.get(sector, pd.DataFrame())
    merged = cds.join(rv, how="outer").join(macro, how="left").loc[SAMPLE_START:SAMPLE_END]
    merged.insert(0, "Sector", sector)
    merged.index.name = "Date"
    frames.append(merged)

master = pd.concat(frames).sort_index()
col_order = ["Sector", "CDS_level", "CDS_change_5d", "N_constituents",
             "RV_daily", "RV_weekly", "RV_monthly",
             "VIX", "Yield_slope", "BAA10Y"]
master = master[[c for c in col_order if c in master.columns]]

master.to_parquet("Sector_Composites.parquet")